**1. ライブラリのインポートとデータ確認**

まずは分析に必要なライブラリを読み込み、Kaggle環境内の入力ファイルを確認する。今回は可視化のために matplotlib、高度な学習のために LightGBM を導入する。

In [1]:
# ライブラリインポート
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import os

# 入力データのファイルパスを確認
# データの階層構造を把握するために実行
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


/kaggle/input/3tounyoubyou/sample_submission.csv
/kaggle/input/3tounyoubyou/train.csv
/kaggle/input/3tounyoubyou/test.csv
/kaggle/input/tounyoubyou/playground-series-s5e12/sample_submission.csv
/kaggle/input/tounyoubyou/playground-series-s5e12/train.csv
/kaggle/input/tounyoubyou/playground-series-s5e12/test.csv


**2. データの読み込み**

学習データ、テストデータ、および提出用のサンプルファイルを読み込む。

In [2]:
# データの読み込み
train = pd.read_csv("/kaggle/input/3tounyoubyou/train.csv")
test = pd.read_csv("/kaggle/input/3tounyoubyou/test.csv")
sample= pd.read_csv("/kaggle/input/3tounyoubyou/sample_submission.csv")

display(train.head())
display(test.head())

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,111,Female,White,Highschool,Middle,Former,Employed,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,145,Female,White,Highschool,Middle,Never,Unemployed,0,0,0
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,184,Male,White,Highschool,Low,Never,Employed,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,128,Male,White,Graduate,Middle,Former,Employed,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,133,Male,White,Graduate,Low,Current,Unemployed,0,0,0


**3. データの整形（不要な列の削除と変数分離）**

機械学習モデルの学習に先立ち、データの整理を行う。

In [3]:
# idは各データに割り振られた固有の番号
# 糖尿病の診断（予測）には寄与しないため「ID」を削除
train1 = train.drop(columns = ["id"])
test1 = test.drop(columns = ["id"])

# train1を説明変数（X）と目的変数（y）に分離
X = train1.drop(columns = ["diagnosed_diabetes"])
y = train1["diagnosed_diabetes"]

**4. 特徴量エンジニアリング**

モデルの予測精度を高めるため、血圧データから医学的背景に基づいた指標を新しく作成する。

* 脈圧 (Pulse Pressure)とは収縮期血圧と拡張期血圧の差のことを指す。血管の硬さや動脈硬化の指標として、糖尿病リスクの評価に役立つと考えられる。
* 平均血圧 (MAP: Mean Arterial Pressure)とは一拍の心周期における平均的な血圧のことを指す。臓器への血流供給圧力を示し、末梢血管の抵抗を反映する指標となる。


**5. LightGBMによるモデリングとHPチューニング**

特徴量エンジニアリング（新たな特徴量の考案）に行き詰まりを感じたため、アプローチを変え、ハイパーパラメータの微調整によるスコアの底上げを図りました。  
限界まで学習させるために early_stopping_rounds = 200 と長めの設定を採用しました。しかし、長すぎる学習は過学習の危険を伴うため、それに対抗する（正則化を強める）意図で、以下のパラメータを新たに投入しました。

* **過学習抑制のパラメータ追加:**
  * min_sum_hessian_in_leaf(1e-3): 決定木の過度な成長を制限するための正則化。
  * feature_fraction_seed(42), bagging_seed(42), data_random_seed(42): 乱数シードを厳密に固定し、ブレを制御。

これらの工夫により、「限界まで学習させつつ、過学習を抑え込む」というギリギリのチューニングを狙いましたが、Early Stopping = 200 の効果がかなり大きく、過学習を引き起こす原因となりました。

**6. クロスバリデーション**

前回のモデル（5-Fold Stratified K-Fold）からさらに精度を厳密に測り、スコアの限界を突破する意図で、今回は分割数を n_splits=10（10-Fold Stratified K-Fold）に増やして実験を行いました。  
※事後分析：検証用データが全体のわずか10%と少なくなったこの設定が、後に過学習を引き起こす大きな要因となりました。


**7. モデルの評価**

評価指標には、分類性能を示すAUCを採用した。
各Foldにおける検証スコアを確認するとともに、すべてのFoldの予測値を結合したOut-of-Fold（OOF）に対する全体AUCを算出し、これをモデルの最終的な評価スコアとした。

In [2]:
X_all = pd.concat([X, test1], axis=0, ignore_index=True)

X_all = X_all.copy()

# 脈圧（収縮期血圧と拡張期血圧の差）
X_all["pulse_pressure"] = X_all["systolic_bp"] - X_all["diastolic_bp"]
# 平均血圧（Mean Arterial Pressure）
X_all["map"] = X_all["diastolic_bp"] + (X_all["pulse_pressure"] / 3.0)

# ベースラインにてデータクレンジングは特に必要ないと判断した
# よってカテゴリ変数を数値化(ワンホットエンコーディング)のみ行う
X_all = pd.get_dummies(X_all, drop_first=False)

# 処理後、再び学習データとテストデータに分割
X = X_all.iloc[:len(X), :].copy()
X_test = X_all.iloc[len(X):, :].copy()

# 前回(3 notebook_糖尿病)では n_splits=5 であったが 10 に増量
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

# 前回(3 notebook_糖尿病)から min_sum_hessian_in_leaf(1e-3), feature_fraction_seed(42), bagging_seed(42), data_random_seed(42)を追加
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.02,
    "num_leaves": 64,
    "min_data_in_leaf": 50,
    "min_sum_hessian_in_leaf": 1e-3,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.5,
    "lambda_l2": 0.5,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed":42,
    "data_random_seed":42,
    "verbosity": -1,
}

NUM_BOOST_ROUND = 100000

# 前回(3 notebook_糖尿病)では EARLY_STOPPING = 100 であったが200に増量
EARLY_STOPPING = 200

# 10分割交差検証（クロスバリデーション）のループ
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    # 学習用と検証用に分割
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    # LightGBM専用のデータセット形式に変換
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dvalid = lgb.Dataset(X_va, label=y_va)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dtrain, dvalid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING),
            lgb.log_evaluation(200)
        ]
    )

    # 検証データに対する予測値を保存（ベストな決定木の本数を使用）
    oof[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    # テストデータに対する予測値を加算（全foldの平均をとるため後で分割数で割る）
    test_pred += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits

    # 各foldごとのAUCスコアを計算・出力
    fold_auc = roc_auc_score(y_va, oof[va_idx])
    print(f"fold {fold} AUC: {fold_auc:.5f}")

# 全検証データに対する全体のAUCスコアを計算・出力
cv_auc = roc_auc_score(y, oof)
print("CV AUC:", cv_auc)

NameError: name 'pd' is not defined

**8. 提出用ファイルの作成**

当該コンペ規定のフォーマットに従って提出用のCSVファイルを出力する。 出力される予測値が計算上の誤差等で万が一0から1の範囲を逸脱した場合に備え、安全対策としてnp.clip関数を用いて確率値を0以上1以下に補正（クリッピング）した上で出力を行った。

In [5]:
sub = pd.DataFrame({
    "id": test["id"].values,
    "diagnosed_diabetes": np.clip(test_pred, 0, 1)
})

sub.to_csv("04_submission.csv", index=False)
sub.head()

,id,diagnosed_diabetes
0,700000,0.495168
1,700001,0.701560
2,700002,0.774018
3,700003,0.363723
4,700004,0.926249
